In [3]:
import numpy as np 

y_predicted = np.array([1,1,0,0,1])
y_true = np.array([0.30,0.7,1,0,0.5])


In [4]:
def mae(y_true, y_predicted):
    return np.mean(np.abs(y_true - y_predicted))

In [5]:
def mae_manual(y_true, y_predicted):
    n = len(y_true)
    absolute_errors = np.abs(y_true - y_predicted)
    sum_absolute_errors = np.sum(absolute_errors)
    mae_value = sum_absolute_errors / n
    return mae_value

In [6]:
mae(y_true, y_predicted)

np.float64(0.5)

In [7]:
mae_manual(y_true, y_predicted)

np.float64(0.5)

In [8]:
# log loss or binary cross entropy  

log_loss = - (y_true * np.log(y_predicted) + (1 - y_true) * np.log(1 - y_predicted))
log_loss = np.mean(log_loss)
log_loss

/tmp/ipykernel_750/393823061.py:3: RuntimeWarning: divide by zero encountered in log
  log_loss = - (y_true * np.log(y_predicted) + (1 - y_true) * np.log(1 - y_predicted))
/tmp/ipykernel_750/393823061.py:3: RuntimeWarning: invalid value encountered in multiply
  log_loss = - (y_true * np.log(y_predicted) + (1 - y_true) * np.log(1 - y_predicted))


np.float64(nan)

In [9]:
# 1-1 will be 0 and log(0) is undefined, so we can add a small value to predicted probabilities to avoid log(0) error. This is called smoothing. we can use epsilon value to add to predicted probabilities.
epsilon = 1e-15    # 0.000000000000001
y_predicted = np.clip(y_predicted, epsilon, 1 - epsilon)
log_loss = - (y_true * np.log(y_predicted) + (1 - y_true) * np.log(1 - y_predicted))
log_loss = np.mean(log_loss)
log_loss

np.float64(17.2696280766844)

In [10]:
y_predicted_new = [max(i, epsilon) for i in y_predicted]
y_predicted_new

[np.float64(0.999999999999999),
 np.float64(0.999999999999999),
 np.float64(1e-15),
 np.float64(1e-15),
 np.float64(0.999999999999999)]

In [11]:
# we are doing this to avoid log(0) error, so we can also do this for 1 - y_predicted to avoid log(0) error for 1 - y_predicted as well. we can use epsilon value to subtract from 1 - y_predicted to avoid log(0) error for 1 - y_predicted as well.
y_predicted_new = [min(i, 1 - epsilon) for i in y_predicted_new]

y_predicted_new

[np.float64(0.999999999999999),
 np.float64(0.999999999999999),
 np.float64(1e-15),
 np.float64(1e-15),
 np.float64(0.999999999999999)]

In [12]:
np.log(y_predicted_new)

array([-9.99200722e-16, -9.99200722e-16, -3.45387764e+01, -3.45387764e+01,
       -9.99200722e-16])

In [13]:
def log_loss_manual(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i, epsilon) for i in y_predicted]
    y_predicted_new = [min(i, 1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    log_loss = - (y_true * np.log(y_predicted_new) + (1 - y_true) * np.log(1 - y_predicted_new))
    log_loss = np.mean(log_loss)
    return log_loss

In [14]:
log_loss_manual(y_true, y_predicted)

np.float64(17.2696280766844)

In [15]:
# mean squared error (MSE) without using numpy
def mse_manual(y_true, y_predicted):
    n = len(y_true)
    squared_errors = (y_true - y_predicted) ** 2
    sum_squared_errors = np.sum(squared_errors)
    mse_value = sum_squared_errors / n
    return mse_value

In [16]:
mse_manual(y_true, y_predicted)

np.float64(0.36599999999999905)

In [17]:
# mse with numpy
def mse(y_true, y_predicted):
    return np.mean((y_true - y_predicted) ** 2) 

In [18]:
mse(y_true, y_predicted)

np.float64(0.36599999999999905)

In [19]:
# log loss without using numpy
def log_loss_manual(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i, epsilon) for i in y_predicted]
    y_predicted_new = [min(i, 1-epsilon) for i in y_predicted_new]
    log_loss = 0
    n = len(y_true)
    for i in range(n):
        log_loss += - (y_true[i] * np.log(y_predicted_new[i]) + (1 - y_true[i]) * np.log(1 - y_predicted_new[i]))
    log_loss = log_loss / n
    return log_loss

In [20]:
log_loss_manual(y_true, y_predicted)

np.float64(17.2696280766844)

In [21]:
''' let's dive into gradient descent and how it works with mean squared error (MSE) loss function. with an insurance_data.csv file which we'll create using pandas and numpy. we'll create a simple linear regression model 
 to predict the insurance charges based on the age of the policyholder. we'll use gradient descent to optimize the parameters of our linear regression model.'''

" let's dive into gradient descent and how it works with mean squared error (MSE) loss function. with an insurance_data.csv file which we'll create using pandas and numpy. we'll create a simple linear regression model \n to predict the insurance charges based on the age of the policyholder. we'll use gradient descent to optimize the parameters of our linear regression model."

In [22]:
import pandas as pd
import numpy as np  
import tensorflow as tf
import matplotlib.pyplot as plt 
%matplotlib inline

I0000 00:00:1776416817.282988     750 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776416836.932948     750 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776416848.280622     750 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [23]:
df = pd.DataFrame({
    "age" : [22,25,47,52,46,55,23,33,43,24,65],
    'affordibility': [1,0,1,0,1,1,0,1,0,1,1],
    'bought_insurance':[0,0,1,0,1,1,0,0,1,1,0]
})

df

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1
5,55,1,1
6,23,0,0
7,33,1,0
8,43,0,1
9,24,1,1


In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[['age','affordibility']], df.bought_insurance, test_size=0.2, random_state=2)

In [25]:
X_train

,age,affordibility
5,55,1
0,22,1
7,33,1
2,47,1
3,52,0
6,23,0
10,65,1
8,43,0


In [26]:
X_test

,age,affordibility
4,46,1
1,25,0
9,24,1


In [27]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age'] /100
X_test_scaled = X_test.copy()
X_test_scaled['age'] = X_test_scaled['age'] /100

In [28]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(2,), activation='sigmoid', kernel_initializer='ones', bias_initializer = 'zeros')
])

model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1776416859.185615     750 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [29]:
model.fit(X_train_scaled, y_train, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 815ms/step - accuracy: 0.3750 - loss: 0.9464
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3750 - loss: 0.9456
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3750 - loss: 0.9449
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3750 - loss: 0.9441
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3750 - loss: 0.9434


In [30]:
model.evaluate(X_test_scaled, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.6667 - loss: 0.4301


[0.4300934374332428, 0.6666666865348816]

In [31]:
model.predict(X_test_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


array([[0.8096444],
       [0.5606377],
       [0.7736087]], dtype=float32)

In [32]:
y_test

4    1
1    0
9    1
Name: bought_insurance, dtype: int64

In [33]:
coeff, intercept = model.get_weights()

In [34]:
coeff, intercept

(array([[0.9950006],
        [0.9950005]], dtype=float32),
 array([-0.00499949], dtype=float32))

In [35]:
import numpy as np
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [36]:
sigmoid(34)

np.float64(0.9999999999999982)

In [37]:
def prediction_fn (age, affordibility):
    weighted_sum = coeff[0] *age+coeff[1]*affordibility +intercept
    return sigmoid(weighted_sum)

In [38]:
prediction_fn(0.46,1)

array([0.8096444], dtype=float32)

In [39]:
def log_loss(y_true, y_pred):
    epsilon = 1e-15
    y_pred_new = [max(i, epsilon) for i in y_pred]
    y_pred_new = [min(i, 1-epsilon) for i in y_pred_new]
    y_pred_new = np.array(y_pred_new)
    return -np.mean(y_true*np.log(y_pred_new) + (1-y_true)*np.log(1-y_pred_new))

In [40]:
sigmoid(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [41]:
def gradient_descent(age, affordibility, y_true, epochs):
    # w1, w2, bias
    w1=w2=1
    bias=0
    learning_rate = 0.5
    n = len(age)

    for i in range(epochs):
        weighted_sum = w1 * age + w2 * affordibility + bias
        y_pred =sigmoid(weighted_sum)
        loss = log_loss(y_true, y_pred)

        w1d = (1/n) * np.dot((y_pred - y_true), np.transpose(age))
        w2d = (1/n) * np.dot((y_pred - y_true), np.transpose(affordibility))

        biasd =  np.mean(y_pred - y_true)  
        w1 = w1 - learning_rate * w1d
        w2 = w2 - learning_rate * w2d
        bias = bias - learning_rate * biasd

        print(f'epoch: {i}, w1 : {w1}, w2 : {w2}, bias : {bias}, loss: {loss}')

    return w1,w2, bias

In [42]:
gradient_descent(X_train_scaled['age'], X_train_scaled['affordibility'], y_train, 5)

epoch: 0, w1 : 0.9327025237787635, w2 : 0.8725146029684344, bias : -0.17687722761553504, loss: 0.9463567177357726
epoch: 1, w1 : 0.8768308751358048, w2 : 0.7627871440051721, bias : -0.3267078686127667, loss: 0.8498447689046407
epoch: 2, w1 : 0.8318939997422801, w2 : 0.6707126655717593, bias : -0.45088405402844567, loss: 0.7809057773762454
epoch: 3, w1 : 0.7967740565346189, w2 : 0.5948841091970398, bias : -0.5521891401911443, loss: 0.7337983241077826
epoch: 4, w1 : 0.7700472509356251, w2 : 0.5331676081290414, bias : -0.6340421425153573, loss: 0.7026212846080759


(np.float64(0.7700472509356251),
 np.float64(0.5331676081290414),
 np.float64(-0.6340421425153573))

In [43]:
# when we want to stop it on the loss threshold

def gradient_descent(age, affordibility, y_true, epochs, loss_threshold):
    # w1, w2, bias
    w1=w2=1
    bias=0
    learning_rate = 0.5
    n = len(age)

    for i in range(epochs):
        weighted_sum = w1 * age + w2 * affordibility + bias
        y_pred =sigmoid(weighted_sum)
        loss = log_loss(y_true, y_pred)

        w1d = (1/n) * np.dot((y_pred - y_true), np.transpose(age))
        w2d = (1/n) * np.dot((y_pred - y_true), np.transpose(affordibility))

        biasd =  np.mean(y_pred - y_true)  
        w1 = w1 - learning_rate * w1d
        w2 = w2 - learning_rate * w2d
        bias = bias - learning_rate * biasd

        if loss<= loss_threshold:
            break

        print(f'epoch: {i}, w1 : {w1}, w2 : {w2}, bias : {bias}, loss: {loss}')

    return w1,w2, bias

In [45]:
gradient_descent(X_train_scaled['age'], X_train_scaled['affordibility'], y_train, 5, .48)

epoch: 0, w1 : 0.9327025237787635, w2 : 0.8725146029684344, bias : -0.17687722761553504, loss: 0.9463567177357726
epoch: 1, w1 : 0.8768308751358048, w2 : 0.7627871440051721, bias : -0.3267078686127667, loss: 0.8498447689046407
epoch: 2, w1 : 0.8318939997422801, w2 : 0.6707126655717593, bias : -0.45088405402844567, loss: 0.7809057773762454
epoch: 3, w1 : 0.7967740565346189, w2 : 0.5948841091970398, bias : -0.5521891401911443, loss: 0.7337983241077826
epoch: 4, w1 : 0.7700472509356251, w2 : 0.5331676081290414, bias : -0.6340421425153573, loss: 0.7026212846080759


(np.float64(0.7700472509356251),
 np.float64(0.5331676081290414),
 np.float64(-0.6340421425153573))

In [53]:
class myNN:
    def __init__(self):
        self.w1 = 1
        self.w2 = 1
        self.bias =0

    def fit(self, X, y , epochs, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_descent(X['age'], X['affordibility'], y, epochs, loss_threshold)
    def predict(self, X_test):
        weighted_sum = self.w1*X_test['age'] + self.w2*X_test['affordibility'] + self.bias
        return sigmoid(weighted_sum)
    def gradient_descent(self, age, affordibility, y_true, epochs, loss_threshold):
    # w1, w2, bias
        w1=w2=1
        bias=0
        learning_rate = 0.5
        n = len(age)
    
        for i in range(epochs):
            weighted_sum = w1 * age + w2 * affordibility + bias
            y_pred =sigmoid(weighted_sum)
            loss = log_loss(y_true, y_pred)
    
            w1d = (1/n) * np.dot((y_pred - y_true), np.transpose(age))
            w2d = (1/n) * np.dot((y_pred - y_true), np.transpose(affordibility))
    
            biasd =  np.mean(y_pred - y_true)  
            w1 = w1 - learning_rate * w1d
            w2 = w2 - learning_rate * w2d
            bias = bias - learning_rate * biasd
    
            if loss<= loss_threshold:
                break
    
            print(f'epoch: {i}, w1 : {w1}, w2 : {w2}, bias : {bias}, loss: {loss}')
    
        return w1,w2, bias

In [54]:
customModel = myNN()
customModel.fit(X_train_scaled, y_train, 5, .64)

epoch: 0, w1 : 0.9327025237787635, w2 : 0.8725146029684344, bias : -0.17687722761553504, loss: 0.9463567177357726
epoch: 1, w1 : 0.8768308751358048, w2 : 0.7627871440051721, bias : -0.3267078686127667, loss: 0.8498447689046407
epoch: 2, w1 : 0.8318939997422801, w2 : 0.6707126655717593, bias : -0.45088405402844567, loss: 0.7809057773762454
epoch: 3, w1 : 0.7967740565346189, w2 : 0.5948841091970398, bias : -0.5521891401911443, loss: 0.7337983241077826
epoch: 4, w1 : 0.7700472509356251, w2 : 0.5331676081290414, bias : -0.6340421425153573, loss: 0.7026212846080759


In [55]:
customModel.predict(X_test_scaled)

4    0.563000
1    0.391376
9    0.520972
dtype: float64